# ConnectaTel Analysis

The goal of this project is to evaluate the **customer behavior** of ConnectaTel, a telecommunications company in Latin America.

I will be working with data recorded **up to the year 2024**, which will allow me to analyze the business's performance within that period.

For this analysis, I will use three datasets:
- **plans.csv** → information on current plans (price, included minutes, included GB, extra cost)
- **users.csv** → customer information (age, city, registration date, plan, churn)
- **usage.csv** → detailed actual usage of services (calls and messages)

I will **explore**, **clean**, and **analyze** this data to build a **statistical profile** of customers, detect **atypical behaviors**, and create **customer segments**.

This analysis will allow me to **identify consumption patterns**, **design retention strategies**, and **suggest improvements to the plans** offered by the company.

> 💡 Before starting, I will think programmatically: What steps do I need? In what order? What do I want to measure and why?


## 🧩 Step 1: Load and Explore Data

Before cleaning or combining the data, it's essential to familiarize myself with the structure of the three datasets. In this step, I will validate that the files load correctly, understand their columns and data types, and detect potential inconsistencies.

### 1.1 Data Loading and Quick View

**🎯 Objective:** To have all 3 datasets ready in memory, understand their content, and perform a preliminary review.


In [ ]:
# import libraries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os

# 1. Import datasets from google drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Defining the exact path where files are stored
path = '/content/drive/MyDrive/Python (datasets y notebooks)/datasets/Sprint 7/'

# 3. Loading datasets into dataframes
users = pd.read_csv(path + 'users_latam.csv')
usage = pd.read_csv(path + 'usage.csv')
plans = pd.read_csv(path + 'plans.csv')


In [ ]:
# Creating copies of each dataset
users_copy = users.copy()
usage_copy = usage.copy()
plans_copy = plans.copy()

# show the first 5 rows of each dataset
plans.head(5)
users.head(5)
usage.head(5)


### 1.2 Exploring Dataset Structure

**🎯 Objective:** To understand the structure of each dataset, review their number of rows and columns, identify the data types of each column, and detect potential inconsistencies or null values before starting the analysis.


In [ ]:
# check the number of rows and columns of each dataset
print("plans", plans.shape)
print("users", users.shape)
print("usage", usage.shape)


In [ ]:
# inspect each dataset with .info()
plans.info()
users.info()
usage.info()


## 🧩 Step 2: Identifying Data Quality Issues

### 2.1 Reviewing Null Values

**🎯 Objective:** To detect the presence and magnitude of missing values to assess if they affect the analysis or require imputation/deletion.


In [ ]:
# number and % of nulls for users
print(users.isna().sum())
print()
print(users.isna().mean())


In [ ]:
# number and % of nulls for usage
print(usage.isna().sum())
print()
print(usage.isna().mean())


**✍️ Null preliminary diagnosis**

Columns with missing values and their proportion:
- ❗ In `users`: `city` (11.7%), `churn_date` (88.4%)
- ❗ In `usage`: `date` (0.1%), `duration` (55.2%), `length` (44.7%)

Type of null treatment per column:
- `users['city']` → **Keep / Ignore for now**: retain as NaN, not critical for this behavior analysis (further verified in step 2.2).
- `users['churn_date']` → **Keep**: high missingness (88.4%) reflects active subscribers, not a data anomaly.
- `usage['date']` → **Filter / Drop**: negligible (0.1%, 50 rows); valid timestamps are needed for time-based aggregation.
- `usage['duration']` & `usage['length']` → **Flag for investigation**: likely depend on event type (call vs. text).


### 2.2 Detecting Invalid Values and Sentinels

**🎯 Objective:** Identify sentinels — values that shouldn't be in the dataset.


In [ ]:
# explore numeric columns of users
users.describe()


- ✅ `users['user_id']` is a unique identifier with a consistent range; no obvious invalid values.
- ⚠️ `users['age']` has a min value of **-999**, a clear sentinel that needs to be addressed. Max age of 75 is reasonable.


In [ ]:
# explore numeric columns of usage
usage.describe()


- ✅ `usage['id']` and `usage['user_id']` are identifiers with no apparent invalid values.
- ✅ `usage['duration']` and `usage['length']` have plausible ranges (min 0.01; max 19.99 / 400.0). No obvious sentinels; nulls are addressed separately.


In [ ]:
# explore categorical columns of users
print("City value counts:")
print(users['city'].value_counts(dropna=False))
print("\nPlan value counts:")
print(users['plan'].value_counts(dropna=False))


- ❗ `city` contains `'?'` — a sentinel representing an unknown city; needs to be replaced with `pd.NA`.
- ✅ `plan` shows two valid values: `Basico` and `Premium`.


In [ ]:
# explore categorical column of usage
usage['type'].value_counts(dropna=False)


- ✅ `type` contains only `'call'` and `'text'` — no invalid values found.

**✍️ Invalid values / sentinels diagnosis**
- `users['age']`: contains a sentinel value of **-999** → replace with the median age.
- `users['city']`: contains a sentinel value of **'?'** → replace with `pd.NA` so it's treated as a proper missing value.


### 2.3 Date Review and Standardization

**🎯 Objective:** Ensure date columns are correctly formatted and detect out-of-range years that indicate capture errors. The dataset is known to contain data up to 2024.


In [ ]:
# Convert reg_date and date columns to datetime
users_copy['reg_date'] = pd.to_datetime(users_copy['reg_date'], errors='coerce')
usage_copy['date'] = pd.to_datetime(usage_copy['date'], errors='coerce')

users_copy['reg_date'].info()
usage_copy['date'].info()


In [ ]:
# Check the years present in each date column
print(users_copy['reg_date'].dt.year.value_counts().sort_index())
print()
print(usage_copy['date'].dt.year.value_counts().sort_index())


**✍️ Diagnosis and actions to take**

- ❗ `users['reg_date']` contains years 2022, 2023, 2024, **and 2026** (40 rows) — 2026 is an impossible year given the data is specified up to 2024. These should be marked as `pd.NaT`.
- ✅ `usage['date']` only contains 2024, consistent with the stated scope.


## 🧩 Step 3: Basic Data Cleaning

### 3.1 Fix Sentinels and Impossible Dates

**🎯 Objective:** Apply cleaning rules to replace sentinel values and fix impossible dates.


In [ ]:
# Replace -999 with the median of age
age_median = users['age'].median()
users_copy['age'] = users_copy['age'].replace(-999, age_median)
users_copy['age'].describe()


In [ ]:
# Replace '?' with NA in city
users_copy['city'] = users_copy['city'].replace('?', pd.NA)
print(users_copy['city'].value_counts(dropna=False))


In [ ]:
# Mark out-of-range (future) dates as NA
current_year = 2024
users_copy.loc[users_copy['reg_date'].dt.year > current_year, 'reg_date'] = pd.NaT
usage_copy.loc[usage_copy['date'].dt.year > current_year, 'date'] = pd.NaT

print("Users reg_date years after cleaning:")
print(users_copy['reg_date'].dt.year.value_counts(dropna=False).sort_index())
print("\nUsage date years after cleaning:")
print(usage_copy['date'].dt.year.value_counts(dropna=False).sort_index())


### 3.2 Handling Missingness in `duration` and `length`

**🎯 Objective:** Decide what to do with null values according to their proportion and relevance — check whether nulls in `duration`/`length` are Missing At Random (MAR) with respect to `type`.


In [ ]:
# MAR verification for duration
print("Usage type when duration is null:")
print(usage_copy[usage['duration'].isna()]['type'].value_counts(dropna=False))


In [ ]:
# MAR verification for length
print("Usage type when length is null:")
print(usage[usage_copy['length'].isna()]['type'].value_counts(dropna=False))


**✍️ Diagnosis:** When `duration` is null, `type` is always `'text'`; when `length` is null, `type` is always `'call'`. This confirms both are **MAR**, dependent on `type`. **Action:** leave them as-is — they encode meaningful information (call vs. text) rather than true missingness.


## 🧩 Step 4: Summary Statistics of Usage per User

### 4.1 Grouping by Usage Behavior

**🎯 Objective:** Summarize the key usage variables per user to represent actual historical behavior, then merge into a single customer profile table.


In [ ]:
# Auxiliary columns
usage_copy["is_text"] = (usage_copy["type"] == "text").astype(int)
usage_copy["is_call"] = (usage_copy["type"] == "call").astype(int)

# Group information by user
usage_agg = usage_copy.groupby('user_id').agg(
    cant_mensajes=('is_text', 'sum'),
    cant_llamadas=('is_call', 'sum'),
    cant_minutos_llamada=('duration', 'sum')
).reset_index()

usage_agg.head(10)


In [ ]:
# Combine the aggregated table with the users dataset
user_profile = users_copy.merge(usage_agg, on='user_id', how='left')
user_profile.head(5)


### 4.2 Statistical Summary per User during 2024

**🎯 Objective:** Analyze the numeric and categorical columns of the merged profile to identify ranges, extremes, and distributions.


In [ ]:
# Statistical summary of the numeric columns
user_profile.describe()


In [ ]:
# Percentage distribution of the plan type
user_profile['plan'].value_counts(normalize=True)


## 🧩 Step 5: Visualization of Distributions and Outliers

### 5.1 Distribution Visualization

**🎯 Objective:** Visually understand how key usage and customer variables behave, and whether they differ by plan type.


In [ ]:
# Histogram: age
sns.histplot(data=user_profile, x='age', hue='plan', palette=['skyblue', 'green'], kde=True)
plt.title('Distribution of Age by Plan Type')
plt.xlabel('Age')
plt.ylabel('Count')
plt.show()


💡 **Insight:** The age distribution is similar across plans; the Basic plan shows a slightly more pronounced concentration of customers in their 30s and 50s.


In [ ]:
# Histogram: number of messages
fig = plt.figure(figsize=(10, 6))
sns.histplot(data=user_profile, x='cant_mensajes', hue='plan', palette=['skyblue', 'green'], kde=True)
plt.title('Distribution of Messages Sent by Plan Type')
plt.xlabel('Number of Messages')
plt.ylabel('Count')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


💡 **Insight:** Both plans show a right-skewed distribution of messages sent — most users send few messages, with Premium showing a slightly wider spread and some heavier senders.


In [ ]:
# Histogram: number of calls
sns.histplot(data=user_profile, x='cant_llamadas', hue='plan', palette=['skyblue', 'green'], kde=True)
plt.title('Distribution of # calls by Plan Type')
plt.xlabel('Number of calls')
plt.ylabel('Count')
plt.show()


💡 **Insight:** Right-skewed for both plans — most users make around 4 calls, very few exceed 6-8. Premium shows a slightly wider spread.


In [ ]:
# Histogram: total call minutes
sns.histplot(data=user_profile, x='cant_minutos_llamada', hue='plan', palette=['skyblue', 'green'], kde=True)
plt.title('Distribution of total minutes per call by Plan Type')
plt.xlabel('Total minutes per call')
plt.ylabel('Count')
plt.show()


💡 **Insight:** Right-skewed distribution; most users accumulate 15-25 total call minutes, with a Premium long tail of heavy users logging over 100 minutes.


### 5.2 Outlier Identification

**🎯 Objective:** Detect extreme values in key usage/customer variables using boxplots, the IQR method, and Z-scores, and decide how to treat them.


In [ ]:
# Boxplots for outlier visualization
columnas_numericas = ['age', 'cant_mensajes', 'cant_llamadas', 'cant_minutos_llamada']

for col in columnas_numericas:
    plt.figure(figsize=(8, 6))
    sns.boxplot(data=user_profile, x=col, palette=['skyblue', 'green'])
    plt.title(f'Boxplot: {col}')
    plt.show()


💡 **Insight:**
- ✅ `age`: no outliers — well-balanced between 18 and 79 years.
- ❗ `cant_mensajes`: right-tailed outliers — some users send very high volumes of messages.
- ❗ `cant_llamadas`: most users make 2-6 calls; a small group significantly exceeds this.
- ❗ `cant_minutos_llamada`: most durations are under 40 minutes, with upper outliers reaching 100-160 minutes.


In [ ]:
# IQR-based outlier detection
columnas_limites = ['cant_mensajes', 'cant_llamadas', 'cant_minutos_llamada']
comparacion = []

for col in columnas_limites:
    Q1 = user_profile[col].quantile(0.25)
    Q3 = user_profile[col].quantile(0.75)
    IQR = Q3 - Q1
    upper_limit = Q3 + 1.5 * IQR
    max_val = user_profile[col].max()
    count_outliers = (user_profile[col] > upper_limit).sum()
    comparacion.append({
        'Column': col,
        'IQR': round(upper_limit, 2),
        'Max': round(max_val, 2),
        'Number of Outliers': count_outliers,
        'Has Outliers': max_val > upper_limit
    })

df_outliers = pd.DataFrame(comparacion)
df_outliers


In [ ]:
# Alternative method: Z-scores
z_scores = (user_profile[columnas_limites] - user_profile[columnas_limites].mean()) / user_profile[columnas_limites].std()
outliers_por_columna = (z_scores.abs() > 3).sum()
print("Outliers per column with Z-score (|Z| > 3):")
print(outliers_por_columna)


💡 **Insight — decision: keep the outliers.**
- `cant_mensajes`: IQR flags an upper bound of 11.50 messages; Z-score flags 21 extreme points. These represent legitimate "power texters" — a high-engagement segment worth preserving.
- `cant_llamadas`: IQR cutoff at 10.50 calls; Z-score flags 30 extreme users. Retaining them preserves real, plausible high-volume calling behavior.
- `cant_minutos_llamada`: IQR caps at 61.86 minutes; Z-score isolates 47 heavy users logging up to 155 minutes — vital for understanding overage fees and upgrade potential.


## 🧩 Step 6: Customer Segmentation

### 6.1 Segmentation by Usage

**🎯 Objective:** Classify each user into `Low use`, `Medium use`, or `High use` based on calls and messages.


In [ ]:
# Create use_group column
user_profile['use_group'] = np.where(
    (user_profile['cant_llamadas'] < 5) & (user_profile['cant_mensajes'] < 5), 'Low use',
    np.where((user_profile['cant_llamadas'] < 10) & (user_profile['cant_mensajes'] < 10), 'Medium use',
             'High use'))

user_profile.head()


### 6.2 Segmentation by Age

**🎯 Objective:** Classify each user into `Young adult` (<30), `Adult` (30-59), or `Senior adult` (>=60).


In [ ]:
# Create age_group column
user_profile = user_profile.copy()
user_profile['age_group'] = np.where(
    user_profile['age'] < 30, 'Young adult',
    np.where(user_profile['age'] < 60, 'Adult', 'Senior adult'))

user_profile.head()


### 6.3 Visualization of Customer Segmentation

**🎯 Objective:** Visualize the distribution of users across `use_group` and `age_group`, split by plan.


In [ ]:
# Distribution of users by usage group
sns.countplot(data=user_profile, x='use_group', palette=['skyblue', 'green'], hue='plan')
plt.title('Distribution of Users by Usage Group')
plt.xlabel('Usage Group', fontweight='bold')
plt.ylabel('Count')
plt.show()


In [ ]:
# Distribution of users by age group
sns.countplot(data=user_profile, x='age_group', palette=['skyblue', 'green'], hue='plan')
plt.title('Distribution of Users by Age Group')
plt.xlabel('Age Group', fontweight='bold')
plt.ylabel('Count')
plt.show()


## 🧩 Step 7: Executive Insight for Stakeholders

**🎯 Objective:** Translate the analysis findings into actionable business conclusions, focused on segmentation, usage patterns, and commercial opportunities.

### ⚠️ Data Quality Issues Detected
- `users['age']`: an outlier value of **-999** (0.025% of rows) was found and replaced with the median age.
- `users['city']`: **'?'** values (2.4% of rows) were detected and replaced with `pd.NA`.
- `users['reg_date']`: **40 records (1%)** with year 2026 were identified as impossible and replaced with `pd.NaT`.
- `usage['date']`: **50 null values (0.125%)** were found and replaced with `pd.NaT`.
- `usage['duration']` and `usage['length']`: high null rates (~55% and ~45%) were determined to be **Missing At Random** and kept, since their absence indicates the type of usage (call vs. message).

### 🔍 Customer Segments and Their Behavior
**Age segments** — `Young adult` (<30), `Adult` (30-59), `Senior adult` (>=60). `Adults` is the largest group, followed by `Senior adults` and `Young adults`. Both plans appear in every segment, but `Basico` has a higher share in each, especially among `Adults`.

**Usage segments** — `Low use` (calls <5 and messages <5), `Medium use` (calls <10 and messages <10), `High use` (remaining). `Medium use` is the largest segment, followed by `Low use` and `High use`. `Basico` users are the majority everywhere, but `Premium` has a notable presence in `Medium` and `High use`.

### 💰 Most Valuable Segments
- **`High use` + `Premium`** customers generate the most revenue due to high service demand.
- **`Adult` / `Senior adult` + `Premium` and/or `High use`** customers show spending capacity and loyalty.

### 📈 Extreme Usage Patterns (Outliers)
Significant outliers in `cant_mensajes`, `cant_llamadas`, and `cant_minutos_llamada` were **retained** — they represent real "intensive users" whose removal would obscure a high-engagement segment that tests plan limits and drives overage/upgrade revenue.

### 💡 Recommendations
1. **Design Ultra-Premium plans** (VIP / Elite / Corporate / Signature) targeting current outlier users, offering more benefits for higher fees.
2. **Migration incentives** (data/minute bonuses) to move `Medium use` users toward `High use` and eventually `Premium`.
3. **Review the `Basico` plan** to better differentiate it from `Premium` and encourage upgrades among `Medium use` Basico customers.
4. **Deeper `churn_date` analysis** to understand retention drivers and identify areas for service improvement.
